# Notebook 2: Anomaly Detection Model
**Project:** Retail Site Failure Early-Warning System  
**Input:** `data/raw/ga4_daily_metrics.csv`  
**Output:** `outputs/anomaly_scores.csv` + `outputs/anomaly_detection_chart.html`

## Why Isolation Forest + CUSUM together
- **Isolation Forest** — answers: is this day statistically abnormal?
- **CUSUM** — answers: exactly when did the structural break begin?
Together they give the complete diagnostic a QuantumBlack team 
delivers: yes there's an anomaly, and it started at this precise moment.

In [5]:
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = (
    "/home/codespace/.config/gcloud/application_default_credentials.json"
)

import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import ruptures as rpt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

# Load daily metrics from Notebook 1
df_daily = pd.read_csv('../data/raw/ga4_daily_metrics.csv')
df_daily['event_date'] = pd.to_datetime(df_daily['event_date'])
df_daily = df_daily.sort_values('event_date').reset_index(drop=True)

print(f"✓ Libraries loaded and data ready")
print(f"  Days: {len(df_daily)}")
print(f"  Date range: {df_daily['event_date'].min().date()} → "
      f"{df_daily['event_date'].max().date()}")
print(f"  Avg conversion rate: {df_daily['conversion_rate'].mean():.2%}")

✓ Libraries loaded and data ready
  Days: 62
  Date range: 2020-11-02 → 2021-01-31
  Avg conversion rate: 1.11%


In [6]:
def engineer_anomaly_features(df):
    """
    Build features for Isolation Forest.
    Raw CVR alone is insufficient — the model needs:
    - Deviation from recent trend (rolling z-score)
    - Session volume context (low sessions + low CVR = stronger signal)
    - Day-of-week encoding (weekend patterns ≠ anomalies)
    """
    df = df.copy()
    
    df['cvr_rolling_7d_mean'] = (
        df['conversion_rate'].rolling(7, min_periods=3).mean()
    )
    df['cvr_rolling_7d_std'] = (
        df['conversion_rate'].rolling(7, min_periods=3).std()
    )
    df['cvr_deviation'] = (
        (df['conversion_rate'] - df['cvr_rolling_7d_mean']) /
        (df['cvr_rolling_7d_std'] + 1e-8)
    )
    df['sessions_rolling_7d_mean'] = (
        df['total_sessions'].rolling(7, min_periods=3).mean()
    )
    df['sessions_deviation'] = (
        (df['total_sessions'] - df['sessions_rolling_7d_mean']) /
        (df['total_sessions'].rolling(7).std() + 1e-8)
    )
    df['day_of_week'] = df['event_date'].dt.dayofweek
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    df['revenue_per_session'] = (
        df['total_revenue'] / (df['total_sessions'] + 1e-8)
    )
    df['revenue_deviation'] = (
        (df['revenue_per_session'] -
         df['revenue_per_session'].rolling(7, min_periods=3).mean()) /
        (df['revenue_per_session'].rolling(7).std() + 1e-8)
    )
    
    return df.dropna().reset_index(drop=True)


df_features = engineer_anomaly_features(df_daily)
print(f"✓ Features engineered")
print(f"  Input rows:  {len(df_daily)}")
print(f"  Output rows: {len(df_features)} "
      f"(first rows dropped — rolling window warmup)")

✓ Features engineered
  Input rows:  62
  Output rows: 56 (first rows dropped — rolling window warmup)


In [7]:
def fit_isolation_forest(df):
    """
    Isolation Forest chosen over One-Class SVM because:
    - Outputs continuous anomaly score (not just binary) — 
      allows severity ranking for client presentation
    - contamination=0.05 maps directly to expected anomaly rate
    - Scales better on time-series with rolling features
    """
    feature_cols = [
        'conversion_rate', 'cvr_deviation',
        'sessions_deviation', 'revenue_deviation',
        'day_of_week', 'is_weekend'
    ]
    
    X = df[feature_cols].values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    model = IsolationForest(
        n_estimators=200,
        contamination=0.05,
        max_samples='auto',
        random_state=42
    )
    
    df = df.copy()
    df['anomaly_score_raw'] = model.fit_predict(X_scaled)
    df['anomaly_score'] = -model.score_samples(X_scaled)
    df['is_anomaly'] = (df['anomaly_score_raw'] == -1).astype(int)
    df['anomaly_severity'] = pd.cut(
        df['anomaly_score'],
        bins=[0, 0.5, 0.55, 0.6, 1.0],
        labels=['normal', 'watch', 'warning', 'critical']
    )
    
    return df, model, scaler


def fit_cusum(df, n_breakpoints=3):
    """
    Pelt algorithm with RBF cost — sensitive to both mean and 
    variance shifts. Identifies when a regime change began,
    not just that it occurred.
    """
    signal = df['conversion_rate'].values
    algo = rpt.Pelt(model='rbf', min_size=3, jump=1)
    algo.fit(signal)
    breakpoints = algo.predict(n_breakpoints)
    breakpoint_dates = [
        df['event_date'].iloc[bp - 1]
        for bp in breakpoints if bp < len(df)
    ]
    return breakpoints, breakpoint_dates


df_scored, model, scaler = fit_isolation_forest(df_features)
breakpoints, breakpoint_dates = fit_cusum(df_scored)

anomalies = df_scored[df_scored['is_anomaly'] == 1]
print(f"✓ Models fitted")
print(f"  Total days scored: {len(df_scored)}")
print(f"  Anomalies flagged: {len(anomalies)} "
      f"({len(anomalies)/len(df_scored):.1%} of days)")
print(f"\nAnomaly severity breakdown:")
print(df_scored['anomaly_severity'].value_counts())
print(f"\nStructural breakpoints:")
for i, d in enumerate(breakpoint_dates):
    idx = df_scored[df_scored['event_date'] == d].index[0]
    cvr_before = df_scored.loc[
        max(0, idx-7):idx, 'conversion_rate'
    ].mean()
    cvr_after = df_scored.loc[
        idx:min(len(df_scored)-1, idx+7), 'conversion_rate'
    ].mean()
    print(f"  Breakpoint {i+1}: {d.date()} | "
          f"CVR before: {cvr_before:.2%} → after: {cvr_after:.2%} | "
          f"shift: {(cvr_after-cvr_before)/cvr_before:.1%}")

✓ Models fitted
  Total days scored: 56
  Anomalies flagged: 3 (5.4% of days)

Anomaly severity breakdown:
anomaly_severity
normal      29
watch       22
warning      5
critical     0
Name: count, dtype: int64

Structural breakpoints:
  Breakpoint 1: 2020-12-23 | CVR before: 1.47% → after: 0.83% | shift: -43.7%
  Breakpoint 2: 2021-01-18 | CVR before: 0.70% → after: 1.26% | shift: 80.9%


In [10]:
# Executive visualization — action title states the finding
normal = df_scored[df_scored['is_anomaly'] == 0]
anomaly_days = df_scored[df_scored['is_anomaly'] == 1]

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=[
        "Daily Conversion Rate with Anomaly Flags and Structural Breakpoints",
        "Anomaly Score — Higher = More Abnormal"
    ],
    vertical_spacing=0.12,
    row_heights=[0.65, 0.35]
)

fig.add_trace(go.Scatter(
    x=normal['event_date'], y=normal['conversion_rate'],
    mode='lines+markers', name='Normal',
    line=dict(color='#2196F3', width=1.5),
    marker=dict(size=4)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=anomaly_days['event_date'], y=anomaly_days['conversion_rate'],
    mode='markers', name='Anomaly flagged',
    marker=dict(color='#E53935', size=10, symbol='x')
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df_scored['event_date'], y=df_scored['cvr_rolling_7d_mean'],
    mode='lines', name='7-day rolling mean',
    line=dict(color='#43A047', width=1, dash='dot')
), row=1, col=1)

colors_bp = ['#FF6F00', '#7B1FA2', '#1B5E20']
for i, bp_date in enumerate(breakpoint_dates):
    fig.add_vline(
        x=bp_date, line_dash='dash',
        line_color=colors_bp[i % len(colors_bp)],
        line_width=1.5, row=1, col=1
    )

threshold = df_scored[df_scored['is_anomaly'] == 1]['anomaly_score'].min()

fig.add_trace(go.Bar(
    x=df_scored['event_date'],
    y=df_scored['anomaly_score'],
    name='Anomaly score',
    marker_color=df_scored['is_anomaly'].map({0: '#BBDEFB', 1: '#E53935'})
), row=2, col=1)

fig.add_hline(
    y=threshold, line_dash='dash', line_color='#E53935',
    annotation_text=f"Alert threshold ({threshold:.3f})",
    row=2, col=1
)

n_critical = len(df_scored[df_scored['anomaly_severity'] == 'critical'])
n_warning = len(df_scored[df_scored['anomaly_severity'] == 'warning'])

fig.update_layout(
    title=dict(
        text=(
            f"<b>{len(breakpoint_dates)} structural breaks detected — "
            f"{n_critical} critical, {n_warning} warning days flagged</b><br>"
            "<sup>Red markers = Isolation Forest anomaly flags · "
            "Dashed lines = CUSUM structural breakpoints</sup>"
        ),
        font=dict(size=14)
    ),
    height=600,
    plot_bgcolor='white',
    paper_bgcolor='white',
    font_family='Arial'
)
fig.update_yaxes(tickformat='.1%', row=1, col=1)

fig.show()
fig.write_html('../outputs/anomaly_detection_chart.html')
print("✓ Chart saved to outputs/anomaly_detection_chart.html")

✓ Chart saved to outputs/anomaly_detection_chart.html


In [11]:
# Export scored dataset for Notebook 3
output_cols = [
    'event_date', 'total_sessions', 'conversions',
    'conversion_rate', 'total_revenue',
    'cvr_rolling_7d_mean', 'cvr_deviation',
    'anomaly_score', 'is_anomaly', 'anomaly_severity'
]

df_scored[output_cols].to_csv('../outputs/anomaly_scores.csv', index=False)
print(f"✓ Anomaly scores exported to outputs/anomaly_scores.csv")
print(f"  Rows: {len(df_scored)}")
print(f"\nFinal severity breakdown:")
print(df_scored['anomaly_severity'].value_counts())

✓ Anomaly scores exported to outputs/anomaly_scores.csv
  Rows: 56

Final severity breakdown:
anomaly_severity
normal      29
watch       22
warning      5
critical     0
Name: count, dtype: int64


## Notebook 2 Complete

**Model decisions:**
- Isolation Forest over One-Class SVM: continuous score, 
  interpretable contamination parameter
- contamination=0.05: ~5% of days expected anomalous
- CUSUM Pelt + RBF: sensitive to mean and variance shifts
- n_breakpoints=3: captures holiday surge + normalization pattern

**Output files:**
- `outputs/anomaly_scores.csv` — all days scored with severity
- `outputs/anomaly_detection_chart.html` — executive visualization

**Next:** `03_root_cause_decomposition.ipynb`